## Network troubleshooting

When packets aren't flowing, the universal tool is **`nicolaka/netshoot`** — a debug image packed with `tcpdump`, `dig`, `curl`, `ip`, `mtr`, `nmap`, and more. Attach it to the *same network* as the broken service and poke from a container's point of view:

```bash
docker run --rm -it --network app-net nicolaka/netshoot
# inside:
dig api                     # does the name resolve? (module 05 embedded DNS)
curl -v http://api/health   # HTTP-level: TLS, redirects, status, timing
ping api                    # basic reachability
tcpdump -i any -nn port 8080   # see the actual packets
```

This walks the stack in order — **name → route → connection → packets** — and each step localises the fault: `dig` fails ⇒ a DNS/wrong-network problem (are both containers on the same user-defined network? module 05); `dig` works but `curl` hangs ⇒ the app isn't listening or a firewall is dropping it; `tcpdump` shows SYNs with no reply ⇒ it's below the app.

**For published ports that aren't reachable from the host**, the suspect is the daemon's own `iptables` rules (module 05's under-the-hood `-p`):

```bash
sudo iptables -t nat -L DOCKER -n -v   # the DNAT rules for -p
sudo iptables -L DOCKER-USER -n -v     # the hook chain for your custom rules
```

A missing `DNAT` rule for your host port means the publish never took effect; a `DROP` in `DOCKER-USER` or an external firewall (ufw/firewalld) is the classic "port works from localhost but not remotely." The through-line for all of module 10's playbooks is one disciplined loop — **reproduce → localise (which layer?) → inspect that layer's state → fix → verify** — and knowing *which* layer (daemon, container, build, network) owns the symptom is what the whole course has been quietly teaching.